# Lab Day 2 — DeepWeeds

Chạy lần lượt theo thứ tự. Chọn cấu hình chỉ bằng validation. Cell test ở Bước 4 có guard chống chạy lặp; chỉ mở cell đó sau khi đã chốt cấu hình.

Gắn dataset ảnh và CSV bằng Kaggle Add Input. Đẩy code lên GitHub trước khi chạy; cell thiết lập clone fork một lần vào `/kaggle/working/lab_repo` và dùng code từ đó. Mọi output được lưu trong thư mục submission của bản clone.


In [ ]:
# Cài dependency. Kaggle Notebook cần bật Internet cho pip và pretrained weights.
!pip -q install timm thop openpyxl

import dataclasses
import hashlib
import json
import subprocess
import sys
from pathlib import Path
from zipfile import ZIP_STORED, ZipFile

import numpy as np
import pandas as pd
import torch
import timm
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

WORKSPACE_DIR = Path("/kaggle/working")
REPO_URL = "https://github.com/VuQuocHuy89/-K4-DAY02-VuQuocHuy-2A202602929.git"
REPO_DIR = WORKSPACE_DIR / "lab_repo"
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
SUBMISSION_DIR = REPO_DIR / "submissions/2A202602929_vu_quoc_huy"
CODE_DIR = SUBMISSION_DIR / "code"
EVAL_FILE = REPO_DIR / "eval.py"
if not CODE_DIR.is_dir() or not EVAL_FILE.is_file():
    raise FileNotFoundError("Repo trên GitHub thiếu code submission hoặc eval.py; hãy push code rồi chạy lại.")
code_hasher = hashlib.sha256()
for module_file in sorted(CODE_DIR.glob("*.py")):
    code_hasher.update(module_file.name.encode())
    code_hasher.update(module_file.read_bytes())
CODE_SHA256 = code_hasher.hexdigest()
# Nếu mở một phiên Kaggle mới, gắn lab_resume.zip của phiên trước qua Add Input.
RESUME_ARCHIVE = None  # ví dụ: Path("/kaggle/input/my-run/lab_resume.zip")
if RESUME_ARCHIVE is not None:
    with ZipFile(RESUME_ARCHIVE) as previous:
        manifest = json.loads(previous.read("lab_resume_manifest.json"))
        if manifest["code_sha256"] != CODE_SHA256:
            raise RuntimeError("Resume archive được tạo bởi phiên bản code khác; không trộn kết quả.")
        if manifest["torch"] != torch.__version__ or manifest["timm"] != timm.__version__:
            raise RuntimeError("Phiên bản torch/timm khác archive; cần môi trường giống phiên trước.")
        allowed = {"runs", "predictions", "curves", "report_assets", "eval_out"}
        for item in previous.infolist():
            parts = Path(item.filename).parts
            if item.filename == "lab_resume_manifest.json":
                continue
            if len(parts) < 4 or parts[:2] != ("submissions", "2A202602929_vu_quoc_huy") or parts[2] not in allowed or ".." in parts:
                raise RuntimeError(f"Invalid resume archive member: {item.filename}")
        previous.extractall(REPO_DIR)
    print("Restored experiment state from:", RESUME_ARCHIVE)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(CODE_DIR))

# Dữ liệu hiện có trong Kaggle Input:
DATA_ROOT = Path("/kaggle/input/datasets/quchuy2k4")
IMAGES_DIR = DATA_ROOT / "images"
LABELS_DIR = DATA_ROOT / "files-csv/file-csv"

RUNS_DIR = SUBMISSION_DIR / "runs"
PRED_DIR = SUBMISSION_DIR / "predictions"
CURVE_DIR = SUBMISSION_DIR / "curves"
ASSET_DIR = SUBMISSION_DIR / "report_assets"
for directory in (RUNS_DIR, PRED_DIR, CURVE_DIR, ASSET_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def save_resume_archive():
    archive_path = WORKSPACE_DIR / "lab_resume.zip"
    temporary_path = WORKSPACE_DIR / "lab_resume.tmp"
    count = 0
    with ZipFile(temporary_path, "w", compression=ZIP_STORED) as archive:
        archive.writestr("lab_resume_manifest.json", json.dumps({
            "commit": COMMIT, "code_sha256": CODE_SHA256,
            "torch": torch.__version__, "timm": timm.__version__,
        }, indent=2))
        for folder in (RUNS_DIR, PRED_DIR, CURVE_DIR, ASSET_DIR, SUBMISSION_DIR / "eval_out"):
            if not folder.is_dir():
                continue
            for file_path in folder.rglob("*"):
                if not file_path.is_file() or file_path.suffix == ".tmp" or "__pycache__" in file_path.parts:
                    continue
                # Completed runs need best.pt + summary, not the large optimizer state.
                if file_path.name == "last.pt" and (file_path.parent / "summary.json").is_file():
                    continue
                archive.write(file_path, file_path.relative_to(REPO_DIR))
                count += 1
    temporary_path.replace(archive_path)
    print("Resume archive:", archive_path, "files:", count, "GB:", round(archive_path.stat().st_size / 1e9, 2))
    return archive_path

import dataset
import inference
import losses
import model as model_lib
import train
import benchmark
import results
import eval as eval_api

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64  # Nếu preflight báo OOM, giảm xuống 32 trước khi train thật.
BACKBONE_EPOCHS = 12  # Bước 1: cùng số epoch cho mọi backbone.
ABLATION_EPOCHS = 12  # Bước 2: cùng số epoch cho T00-T08.
FINAL_EPOCHS = 20   # Bước 4: final và baseline cùng số epoch.
BACKBONE_CANDIDATES = [
    "resnet50", "convnext_tiny", "deit_small_patch16_224",
    "efficientnet_b0", "mobilenetv3_large_100",
]
print("Code source:", REPO_URL, "| commit:", COMMIT)
print("Output:", SUBMISSION_DIR)
print("Images:", IMAGES_DIR, "| exists:", IMAGES_DIR.is_dir())
print("Labels:", LABELS_DIR, "| exists:", LABELS_DIR.is_dir())
print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
print("GPU count:", torch.cuda.device_count(), "| training device:", DEVICE)
if torch.cuda.is_available():
    print("cuda:0 memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
environment = {
    "git_commit": COMMIT, "code_sha256": CODE_SHA256,
    "batch_size": BATCH_SIZE,
    "backbone_epochs": BACKBONE_EPOCHS, "ablation_epochs": ABLATION_EPOCHS,
    "final_epochs": FINAL_EPOCHS,
    "python": sys.version.split()[0], "torch": torch.__version__,
    "torchvision": torchvision.__version__, "timm": timm.__version__,
    "numpy": np.__version__, "pandas": pd.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "gpu_count": torch.cuda.device_count(),
}
(SUBMISSION_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
print("Environment:", environment)


## Bước 0 — Kiểm tra dữ liệu và EDA

Đọc split fold 0, ghi kiểm tra vào output và vẽ phân bố lớp cùng ảnh mẫu. Xem ảnh có đúng nhãn không trước khi train.


In [ ]:
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_report = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)

labels_all = pd.read_csv(LABELS_DIR / "labels.csv")
class_names = (
    labels_all.drop_duplicates("Label")
    .set_index("Label")["Species"]
    .to_dict()
)
print("Label mapping:", class_names)
print("Split counts:", split_report["counts"])
print("Missing images:", split_report["missing_images"])
train_counts = train_df["Label"].value_counts()
imbalance_ratio = float(train_counts.max() / train_counts.min())
example_files = train_df.groupby("Label").head(3)["Filename"].tolist()
image_shapes, image_modes = set(), set()
for filename in example_files:
    with Image.open(IMAGES_DIR / str(filename)) as image:
        image_shapes.add(tuple(image.size))
        image_modes.add(image.mode)
eda_summary = {
    "train_class_imbalance_max_over_min": imbalance_ratio,
    "sample_image_sizes": sorted([list(size) for size in image_shapes]),
    "sample_image_modes": sorted(image_modes),
    "sampled_images_per_class": 3,
}
(ASSET_DIR / "eda_summary.json").write_text(json.dumps(eda_summary, indent=2))
print("Train imbalance (largest/smallest class):", round(imbalance_ratio, 2))
print("Sample image dimensions/modes:", image_shapes, image_modes)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name, frame in zip(
    axes, ("Train", "Validation", "Test"), (train_df, val_df, test_df)
):
    counts = frame["Label"].value_counts().reindex(sorted(class_names), fill_value=0)
    ax.bar([class_names[i] for i in counts.index], counts.values)
    ax.set_title(f"{name}: {len(frame)} images")
    ax.set_ylabel("Number of images")
    ax.tick_params(axis="x", rotation=75)
fig.tight_layout()
fig.savefig(ASSET_DIR / "class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(9, 3, figsize=(10, 24))
for row, label_id in enumerate(sorted(class_names)):
    samples = train_df[train_df["Label"] == label_id].sample(n=3, random_state=42)
    for col, (_, sample) in enumerate(samples.iterrows()):
        with Image.open(IMAGES_DIR / str(sample["Filename"])) as image:
            axes[row, col].imshow(image.convert("RGB"))
        axes[row, col].set_title(f"{label_id}: {class_names[label_id]}")
        axes[row, col].axis("off")
fig.tight_layout()
fig.savefig(ASSET_DIR / "sample_images.png", dpi=160, bbox_inches="tight")
plt.show()


## Kiểm tra pipeline trước khi train

Smoke test tạo batch, xác nhận shape và nhãn, kiểm tra focal loss gamma=0, Mixup/CutMix, loss ban đầu và khả năng overfit batch nhỏ.


In [ ]:
train_loader_smoke = dataset.make_loader(
    train_df, IMAGES_DIR,
    dataset.build_transforms(train=True, img_size=224, aug="basic"),
    batch_size=4, train=True, num_workers=2,
)
x, y, names = next(iter(train_loader_smoke))
print("Images:", tuple(x.shape), "| labels:", y.tolist(), "| files:", names)
assert tuple(x.shape) == (4, 3, 224, 224)
assert int(y.min()) >= 0 and int(y.max()) <= 8 and len(names) == 4

# Focal gamma=0 phải bằng CE khi không có alpha/class weights.
logits_check = torch.randn(8, 9)
labels_check = torch.randint(0, 9, (8,))
ce_value = losses.build_criterion("ce")(logits_check, labels_check)
focal_zero = losses.build_criterion("focal", gamma=0)(logits_check, labels_check)
print("CE:", float(ce_value), "| focal gamma=0:", float(focal_zero))
assert torch.allclose(ce_value, focal_zero, atol=1e-6)

for mix_mode in ("mixup", "cutmix"):
    mixed_x, mixed_targets = losses.mix_batch(x, y, alpha=1.0, mode=mix_mode)
    print(mix_mode, tuple(mixed_x.shape), "lambda:", mixed_targets[2])
    assert mixed_x.shape == x.shape

# Kiểm tra loss khởi đầu và overfit một batch nhỏ trên model ngẫu nhiên.
torch.manual_seed(7)
check_model = model_lib.build_model(
    "resnet18", pretrained=False, num_classes=9, init="scratch"
).to(DEVICE)
x_device, y_device = x.to(DEVICE), y.to(DEVICE)
check_criterion = torch.nn.CrossEntropyLoss()
check_model.train()
with torch.no_grad():
    initial_loss = float(check_criterion(check_model(x_device), y_device).item())
print("Initial CE loss (reference ln(9)=2.197):", initial_loss)

check_optimizer = torch.optim.AdamW(check_model.parameters(), lr=1e-2, weight_decay=0.0)
for _ in range(150):
    check_model.train()
    check_optimizer.zero_grad(set_to_none=True)
    loss_value = check_criterion(check_model(x_device), y_device)
    loss_value.backward()
    check_optimizer.step()
check_model.eval()
with torch.inference_mode():
    final_loss = float(check_criterion(check_model(x_device), y_device).item())
print("Same-batch loss after 150 steps:", final_loss)
assert final_loss < 0.5, "Model failed to overfit the fixed mini-batch; inspect labels/model/loss"
# Fused Conv-BN should preserve eval outputs to numerical tolerance.
conv_bn = torch.nn.Sequential(
    torch.nn.Conv2d(3, 8, kernel_size=3, padding=1, bias=False),
    torch.nn.BatchNorm2d(8),
).eval()
fusion_input = torch.randn(2, 3, 16, 16)
fused_conv_bn = inference.fuse_conv_bn(conv_bn)
with torch.inference_mode():
    before_fusion = conv_bn(fusion_input)
    after_fusion = fused_conv_bn(fusion_input)
assert torch.allclose(before_fusion, after_fusion, atol=1e-5, rtol=1e-4)
print("Conv-BN fusion output check passed")

del check_model, check_optimizer, conv_bn, fused_conv_bn, x_device, y_device, loss_value
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Thử một optimizer step đúng batch size trên từng backbone trước khi train dài.
if DEVICE.type != "cuda":
    raise RuntimeError("Cần bật GPU Kaggle trước khi huấn luyện.")
def probe_backbone_memory(backbone_name):
    candidate = model_lib.build_model(backbone_name, pretrained=False, num_classes=9).to(DEVICE)
    probe_cfg = train.Config(backbone=backbone_name, batch_size=BATCH_SIZE)
    optimizer = train.build_optimizer(candidate, probe_cfg)
    scaler = train._make_scaler(True)
    inputs = torch.randn(BATCH_SIZE, 3, 224, 224, device=DEVICE)
    labels = torch.randint(0, 9, (BATCH_SIZE,), device=DEVICE)
    candidate.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        probe_loss = torch.nn.functional.cross_entropy(candidate(inputs).float(), labels)
    scaler.scale(probe_loss).backward()
    scaler.step(optimizer)
    scaler.update()
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated(DEVICE) / 1024**3

preflight_memory_gb = {}
for backbone_name in BACKBONE_CANDIDATES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    try:
        peak_gb = probe_backbone_memory(backbone_name)
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise RuntimeError(
            f"{backbone_name} OOM với batch={BATCH_SIZE}; giảm BATCH_SIZE xuống 32 ở cell đầu rồi chạy lại trước khi train."
        ) from exc
    print(f"Preflight {backbone_name}: peak {peak_gb:.2f} GiB trên cuda:0")
    preflight_memory_gb[backbone_name] = peak_gb
    if peak_gb > 0.85 * torch.cuda.get_device_properties(0).total_memory / 1024**3:
        print("Cảnh báo: VRAM sát giới hạn; cân nhắc BATCH_SIZE=32 trước khi train.")
torch.cuda.empty_cache()
# Tải trước toàn bộ trọng số để phát hiện lỗi mạng/model trước khi mất giờ GPU.
for backbone_name in BACKBONE_CANDIDATES:
    try:
        pretrained_probe = model_lib.build_model(backbone_name, pretrained=True, num_classes=9)
    except Exception as exc:
        raise RuntimeError(f"Không tải được pretrained weights của {backbone_name}") from exc
    print("Pretrained:", backbone_name, getattr(pretrained_probe, "_lab_pretrained_tag", "unknown"))
    del pretrained_probe
(ASSET_DIR / "preflight_memory.json").write_text(json.dumps(preflight_memory_gb, indent=2))


## Bước 1 — Sàng lọc backbone

Mỗi model chạy cùng cấu hình nền, cùng fold và seed 0. Để đo tốc độ trước, có thể tạm đặt `BACKBONES = ["resnet50"]`, chạy B01, rồi khôi phục danh sách đủ năm model và chạy lại cell; B01 có summary phù hợp sẽ được dùng lại. Sau khi đủ năm, xem ước lượng thời gian còn lại trước Bước 2. Cell lưu checkpoint trong `lab_resume.zip`.


In [ ]:
BACKBONES = list(BACKBONE_CANDIDATES)

def make_config(exp_id, backbone, seed=0, **overrides):
    base = dict(
        exp_id=exp_id,
        seed=seed,
        fold=0,
        backbone=backbone,
        init="finetune",
        img_size=224,
        aug="basic",
        loss="ce",
        epochs=BACKBONE_EPOCHS,
        batch_size=BATCH_SIZE,
        lr_backbone=1e-4,
        lr_head=1e-3,
        weight_decay=0.05,
        warmup_epochs=1.0,
        amp=True,
        num_workers=2,
        images_dir=str(IMAGES_DIR),
        labels_dir=str(LABELS_DIR),
        out_dir=str(RUNS_DIR),
        pred_dir=str(PRED_DIR),
        curve_dir=str(CURVE_DIR),
        save_test_predictions=False,
    )
    base.update(overrides)
    return train.Config(**base)

def run_or_load(cfg):
    summary_path = train.run_dir(cfg) / "summary.json"
    if not summary_path.is_file():
        return train.run(cfg)
    config_path = train.run_dir(cfg) / "config.json"
    saved = json.loads(config_path.read_text())
    expected = json.loads(json.dumps(dataclasses.asdict(cfg)))
    mismatch = [key for key, value in expected.items() if saved.get(key) != value]
    if mismatch:
        raise RuntimeError(
            f"Saved {cfg.exp_id} run differs in {mismatch}; use a new exp_id instead of mixing results"
        )
    print("Using completed matching run:", cfg.exp_id, cfg.seed)
    return json.loads(summary_path.read_text())

backbone_results = []
for index, backbone_name in enumerate(BACKBONES, start=1):
    exp_id = f"B{index:02d}"
    cfg = make_config(exp_id, backbone_name)
    result = run_or_load(cfg)
    # Batch-1 latency is measured once for every screened backbone.
    latency_path = RUNS_DIR / exp_id / "seed0" / "latency_batch1.json"
    if latency_path.is_file():
        backbone_latency = json.loads(latency_path.read_text())
    else:
        checkpoint = torch.load(
            RUNS_DIR / exp_id / "seed0" / "best.pt", map_location="cpu", weights_only=False
        )
        screen_model = model_lib.build_model(
            result["backbone"], pretrained=False, num_classes=9,
            init=result.get("init", "finetune"),
        )
        screen_model.load_state_dict(checkpoint["state_dict"])
        screen_model = screen_model.to(DEVICE).eval()
        backbone_latency = benchmark.latency_report(
            screen_model, batch_size=1, img_size=224, dtype="fp32",
            device=str(DEVICE), warmup=10, iters=50,
        )
        latency_path.write_text(json.dumps(backbone_latency, indent=2))
        del screen_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    result["latency_batch1_ms"] = backbone_latency["p50"]
    result["latency_batch1_p95_ms"] = backbone_latency["p95"]
    result["gpu"] = backbone_latency["gpu"]
    class_metrics = eval_api.load_group(
        str(PRED_DIR / f"{exp_id}_seed0_val.csv"),
        str(LABELS_DIR / "val_subset0.csv"), ref_what="val",
    ).metrics[0]
    result["chinee_recall_val"] = float(class_metrics["recall"][dataset.CLASS_NAMES.index("Chinee Apple")])
    result["snake_recall_val"] = float(class_metrics["recall"][dataset.CLASS_NAMES.index("Snake Weed")])
    backbone_results.append(result)

backbone_table = pd.DataFrame(backbone_results).sort_values("macro_f1_val", ascending=False)
display(backbone_table[[
    "exp_id", "backbone", "pretrained_tag", "params_m", "gmacs",
    "macro_f1_val", "top1_val", "chinee_recall_val", "snake_recall_val",
    "mean_epoch_seconds", "latency_batch1_ms"
]])
backbone_table.to_csv(SUBMISSION_DIR / "backbone_screen.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(backbone_table["latency_batch1_ms"], backbone_table["macro_f1_val"])
for _, row in backbone_table.iterrows():
    ax.annotate(row["backbone"], (row["latency_batch1_ms"], row["macro_f1_val"]))
ax.set_xlabel("Batch-1 model latency p50 (ms, FP32)")
ax.set_ylabel("Validation macro-F1")
ax.set_title("Backbone quality and latency")
fig.tight_layout()
fig.savefig(ASSET_DIR / "backbone_accuracy_latency.png", dpi=160)
plt.show()
selected_epoch_seconds = float(backbone_table.iloc[0]["mean_epoch_seconds"])
projected_remaining_hours = (9 * ABLATION_EPOCHS + 6 * FINAL_EPOCHS) * selected_epoch_seconds / 3600
print(f"Ước lượng thô phần train còn lại trên 1 GPU: {projected_remaining_hours:.1f} giờ; chưa tính inference và T07 EMA.")
save_resume_archive()  # Tải bản này nếu cần chuyển sang phiên Kaggle mới.


## Bước 2 — Ablation công thức huấn luyện

Chọn backbone dựa trên validation và chi phí. T01–T07 mỗi lần chỉ đổi một yếu tố so với T00: khởi tạo, augmentation, loss/cân bằng lớp và EMA. T08 kết hợp hai yếu tố chọn từ validation. Giữ cùng backbone, split, seed và số epoch cho mọi lượt T.


In [ ]:
# Chọn theo validation macro-F1; chỉ sửa override nếu bảng macro-F1/latency
# cho thấy một lựa chọn cân bằng hơn, và ghi rõ lý do trong báo cáo.
BACKBONE_OVERRIDE = None
SELECTED_BACKBONE = BACKBONE_OVERRIDE or str(backbone_table.iloc[0]["backbone"])
print("Backbone cho ablation/final:", SELECTED_BACKBONE)

def make_recipe(exp_id, **overrides):
    return make_config(
        exp_id, SELECTED_BACKBONE, epochs=ABLATION_EPOCHS,
        pred_dir=str(PRED_DIR / "screen"), curve_dir=str(CURVE_DIR / "screen"),
        **overrides
    )

single_factor_configs = [
    make_recipe("T00"),  # CE + pretrained + basic augmentation
    make_recipe("T01", init="scratch"),
    make_recipe("T02", init="frozen"),
    make_recipe("T03", mix="cutmix"),
    make_recipe("T04", aug="color"),
    make_recipe("T05", loss="focal", focal_gamma=2.0),
    make_recipe("T06", loss="ce_weighted", class_weight_beta=0.999),
    make_recipe("T07", ema_decay=0.995),
]
recipe_results = []
for cfg in single_factor_configs:
    result = run_or_load(cfg)
    recipe_results.append(result)

# Kết hợp một augmentation với yếu tố loss/EMA tốt nhất theo validation.
by_id = {row["exp_id"]: row for row in recipe_results}
config_by_id = {cfg.exp_id: cfg for cfg in single_factor_configs}
best_aug_id = max(("T03", "T04"), key=lambda key: by_id[key]["macro_f1_val"])
best_other_id = max(("T05", "T06", "T07"), key=lambda key: by_id[key]["macro_f1_val"])
COMBO_SOURCE_IDS = (best_aug_id, best_other_id)
combo_overrides = {}
for source_id in COMBO_SOURCE_IDS:
    source = config_by_id[source_id]
    for field in ("aug", "mix", "loss", "class_weight_beta", "ema_decay"):
        value = getattr(source, field)
        if value != getattr(single_factor_configs[0], field):
            combo_overrides[field] = value
combo_cfg = make_recipe("T08", **combo_overrides)
recipe_configs = single_factor_configs + [combo_cfg]
recipe_results.append(run_or_load(combo_cfg))
print("T08 combines validation candidates:", COMBO_SOURCE_IDS)

for result in recipe_results:
    class_metrics = eval_api.load_group(
        str(PRED_DIR / "screen" / f"{result['exp_id']}_seed0_val.csv"),
        str(LABELS_DIR / "val_subset0.csv"), ref_what="val",
    ).metrics[0]
    result["chinee_recall_val"] = float(class_metrics["recall"][dataset.CLASS_NAMES.index("Chinee Apple")])
    result["snake_recall_val"] = float(class_metrics["recall"][dataset.CLASS_NAMES.index("Snake Weed")])
recipe_table = pd.DataFrame(recipe_results).sort_values("macro_f1_val", ascending=False)
display(recipe_table[[
    "exp_id", "backbone", "init", "best_epoch", "macro_f1_val",
    "top1_val", "chinee_recall_val", "snake_recall_val",
    "ece_val", "mean_epoch_seconds"
]])
recipe_table.to_csv(SUBMISSION_DIR / "training_screen.csv", index=False)

# Chốt recipe bằng validation, cân nhắc cả recall hai lớp khó và chi phí.
RECIPE_OVERRIDE = None
BEST_RECIPE_ID = RECIPE_OVERRIDE or str(recipe_table.iloc[0]["exp_id"])
if BEST_RECIPE_ID not in {cfg.exp_id for cfg in recipe_configs}:
    raise ValueError("RECIPE_OVERRIDE không có trong các lượt T00-T08")
BEST_RECIPE_RESULT = next(r for r in recipe_results if r["exp_id"] == BEST_RECIPE_ID)
BEST_RECIPE_CONFIG = next(c for c in recipe_configs if c.exp_id == BEST_RECIPE_ID)
print("Candidate chosen from validation:", BEST_RECIPE_ID, BEST_RECIPE_CONFIG)
save_resume_archive()


## Bước 3 — Inference, hiệu chuẩn và latency

Thử trên validation, không train lại và không mở test. I01 lật ngang, I02 5 crop, I03 gộp xác suất/logit, I07 temperature scaling. Đo latency với warmup và ít nhất 50 lần.


In [ ]:
best_checkpoint_path = RUNS_DIR / BEST_RECIPE_ID / "seed0" / "best.pt"
checkpoint = torch.load(best_checkpoint_path, map_location="cpu", weights_only=False)
best_model = model_lib.build_model(
    BEST_RECIPE_CONFIG.backbone,
    pretrained=False,  # checkpoint supplies all trained/pretrained weights; avoid a redundant download
    num_classes=9,
    drop_rate=BEST_RECIPE_CONFIG.drop_rate,
    init=BEST_RECIPE_CONFIG.init,
)
best_model.load_state_dict(checkpoint["state_dict"])
best_model = best_model.to(DEVICE).eval()

val_loader_224 = dataset.make_loader(
    val_df, IMAGES_DIR, dataset.build_transforms(False, 224),
    batch_size=BATCH_SIZE, train=False, num_workers=2,
)
val_loader_256 = dataset.make_loader(
    val_df, IMAGES_DIR, dataset.build_transforms(False, 256),
    batch_size=BATCH_SIZE, train=False, num_workers=2,
)

def metric_row(exp_id, method, filenames, labels, probs, views, latency=None, notes=""):
    metrics = eval_api.compute_metrics(labels, probs.argmax(axis=1), probs)
    return {
        "exp_id": exp_id, "method": method, "checkpoint": BEST_RECIPE_ID,
        "views_or_models": views, "macro_f1_val": metrics["macro_f1"],
        "top1_val": metrics["top1"], "ece_val": metrics["ece"],
        "p50_ms_batch1": None if latency is None else latency["p50"],
        "p95_ms_batch1": None if latency is None else latency["p95"],
        "p99_ms_batch1": None if latency is None else latency["p99"],
        "images_per_s": None if latency is None else latency["images_per_s"],
        "relative_cost_vs_I00": None, "notes": notes,
    }

base_names, base_labels, base_logits = inference.predict_logits(
    best_model, val_loader_224, DEVICE
)
temperature_val = inference.fit_temperature(base_logits, base_labels)
prob_i00 = inference.apply_temperature(base_logits, 1.0)
prob_i07 = inference.apply_temperature(base_logits, temperature_val)

names_i01, labels_i01, prob_i01 = inference.predict_probabilities(
    best_model, val_loader_224, DEVICE, method="I01"
)
names_i02, labels_i02, prob_i02 = inference.predict_probabilities(
    best_model, val_loader_256, DEVICE, method="I02", crop_size=224
)
_, _, prob_i03_prob = inference.predict_probabilities(
    best_model, val_loader_224, DEVICE, method="I03", aggregation="prob"
)
_, _, prob_i03_logit = inference.predict_probabilities(
    best_model, val_loader_224, DEVICE, method="I03", aggregation="logit"
)

inference_results = [
    metric_row("I00", "1-view", base_names, base_labels, prob_i00, 1),
    metric_row("I01", "horizontal flip TTA", names_i01, labels_i01, prob_i01, 2),
    metric_row("I02", "five-crop TTA", names_i02, labels_i02, prob_i02, 5),
    metric_row("I03_prob", "average probabilities", base_names, base_labels, prob_i03_prob, 2),
    metric_row("I03_logit", "average logits", base_names, base_labels, prob_i03_logit, 2),
    metric_row("I07", "temperature scaling", base_names, base_labels, prob_i07, 1,
               notes=f"T fitted on val: {temperature_val:.5f}"),
]
inference_table = pd.DataFrame(inference_results)

latency_i00 = benchmark.latency_report(
    best_model, batch_size=1, img_size=224, dtype="fp32",
    device=str(DEVICE), warmup=10, iters=50,
)
latency_i01 = benchmark.tta_latency(
    best_model, k_views=2, batch_size=1, img_size=224, dtype="fp32",
    device=str(DEVICE), warmup=10, iters=50,
)
latency_i02 = benchmark.tta_latency(
    best_model, k_views=5, batch_size=1, img_size=224, dtype="fp32",
    device=str(DEVICE), warmup=10, iters=50,
)
# Measure throughput at a larger batch, falling back to 16 only if batch 32
# exceeds the available GPU memory.
try:
    latency_batch_large = benchmark.latency_report(
        best_model, batch_size=32, img_size=224, dtype="fp32",
        device=str(DEVICE), warmup=10, iters=50,
    )
except RuntimeError as exc:
    if "out of memory" not in str(exc).lower():
        raise
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Batch 32 ran out of memory; retrying batch 16")
    latency_batch_large = benchmark.latency_report(
        best_model, batch_size=16, img_size=224, dtype="fp32",
        device=str(DEVICE), warmup=10, iters=50,
    )

fused_model = inference.fuse_conv_bn(best_model).to(DEVICE).eval()
latency_fused = benchmark.latency_report(
    fused_model, batch_size=1, img_size=224, dtype="fp32",
    device=str(DEVICE), warmup=10, iters=50,
)
latency_results = [
    dict(configuration="I00", **latency_i00),
    dict(configuration="I01", **latency_i01),
    dict(configuration="I02", **latency_i02),
    dict(configuration="I00-batch-large", **latency_batch_large),
    dict(configuration="I00-BN-fused", **latency_fused),
]

latency_for_method = {
    "I00": latency_i00, "I01": latency_i01, "I02": latency_i02,
    "I03_prob": latency_i01, "I03_logit": latency_i01, "I07": latency_i00,
}
for row in inference_results:
    measured = latency_for_method[row["exp_id"]]
    row["p50_ms_batch1"] = measured["p50"]
    row["p95_ms_batch1"] = measured["p95"]
    row["p99_ms_batch1"] = measured["p99"]
    row["images_per_s"] = measured["images_per_s"]
    row["relative_cost_vs_I00"] = measured["p50"] / latency_i00["p50"]
inference_table = pd.DataFrame(inference_results)
inference_table.to_csv(SUBMISSION_DIR / "inference_screen.csv", index=False)

latency_rows = []
for row in latency_results:
    latency_rows.append({
        "configuration": row["configuration"], "gpu": row["gpu"],
        "dtype": row["dtype"], "batch": row["batch"], "img_size": row["img_size"],
        "bn_fused": row["configuration"] == "I00-BN-fused",
        "p50_ms": row["p50"], "p95_ms": row["p95"], "p99_ms": row["p99"],
        "images_per_s": row["images_per_s"], "torch_version": row["torch"],
        "preprocessing_included": False,
    })
print("Temperature fitted on validation:", temperature_val)
display(inference_table[["exp_id", "method", "macro_f1_val", "top1_val", "ece_val", "p50_ms_batch1", "p95_ms_batch1", "relative_cost_vs_I00"]])
display(pd.DataFrame(latency_results)[["configuration", "gpu", "dtype", "batch", "p50", "p95", "p99", "images_per_s"]])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(inference_table["p95_ms_batch1"], inference_table["macro_f1_val"])
for _, row in inference_table.iterrows():
    ax.annotate(row["exp_id"], (row["p95_ms_batch1"], row["macro_f1_val"]))
ax.set_xlabel("Batch-1 model latency p95 (ms, FP32; preprocessing excluded)")
ax.set_ylabel("Validation macro-F1")
ax.set_title("Inference accuracy and latency")
fig.tight_layout()
fig.savefig(ASSET_DIR / "inference_accuracy_latency.png", dpi=160)
plt.show()


## Bước 4 — Chung kết

Chỉ chạy sau khi đã xem và chốt kết quả validation. Chọn phương pháp inference từ các kết quả I00/I01/I02/I03/I07. Cell sẽ chạy final và T00 baseline cho seed 0,1,2; file predictions tồn tại thì lần chạy đó bị bỏ qua để tránh test lặp.


In [ ]:
# Mặc định chọn theo macro-F1 và ECE validation; I07 là calibration sau inference.
# Nếu chọn một TTA, khớp T riêng trên xác suất validation của phương pháp đó.
INFERENCE_OVERRIDE = None
FINAL_INFERENCE_ID = INFERENCE_OVERRIDE or str(
    inference_table.sort_values(["macro_f1_val", "ece_val"], ascending=[False, True]).iloc[0]["exp_id"]
)
method_map = {
    "I00": ("I00", "prob"),
    "I01": ("I01", "prob"),
    "I02": ("I02", "prob"),
    "I03_prob": ("I03", "prob"),
    "I03_logit": ("I03", "logit"),
    "I07": ("I00", "prob"),  # I07 calibrates the selected one-view inference.
}
validation_probabilities = {
    "I00": prob_i00, "I01": prob_i01, "I02": prob_i02,
    "I03_prob": prob_i03_prob, "I03_logit": prob_i03_logit,
}
if FINAL_INFERENCE_ID not in method_map:
    raise ValueError("Unknown final inference choice")
method_name, aggregation_name = method_map[FINAL_INFERENCE_ID]
if FINAL_INFERENCE_ID == "I07":
    FINAL_TEMPERATURE = temperature_val
else:
    chosen_val_probs = validation_probabilities[FINAL_INFERENCE_ID]
    FINAL_TEMPERATURE = inference.fit_temperature(
        np.log(np.clip(chosen_val_probs, 1e-12, 1.0)), base_labels
    )
print("Final inference chosen on validation:", FINAL_INFERENCE_ID, "method=", method_name,
      "screening T=", FINAL_TEMPERATURE, "(final fits T on each seed's val)")

# Chạy hết các bước validation, xem bảng/đồ thị và chốt lựa chọn trước khi mở test.
RUN_FINAL_TEST = False
if not RUN_FINAL_TEST:
    raise RuntimeError("Đã dừng trước test. Sau khi chốt bằng validation, đặt RUN_FINAL_TEST=True và chạy lại cell này.")

final_runs = []
for seed in (0, 1, 2):
    final_cfg = dataclasses.replace(
        BEST_RECIPE_CONFIG,
        exp_id="F01",
        seed=seed,
        epochs=FINAL_EPOCHS,
        save_test_predictions=True,
        inference_method=method_name,
        inference_aggregation=aggregation_name,
        temperature=None,  # fit trên validation riêng của seed này
        out_dir=str(RUNS_DIR),
        pred_dir=str(PRED_DIR),
        curve_dir=str(CURVE_DIR),
    )
    baseline_cfg = make_config(
        "T00", SELECTED_BACKBONE, seed=seed, save_test_predictions=True,
        epochs=FINAL_EPOCHS,
        inference_method="I00", temperature=1.0,
        out_dir=str(RUNS_DIR / "final_baseline"),
    )

    for cfg in (final_cfg, baseline_cfg):
        test_path = PRED_DIR / f"{cfg.exp_id}_seed{seed}_test.csv"
        if test_path.is_file():
            config_path = train.run_dir(cfg) / "config.json"
            if not config_path.is_file():
                raise RuntimeError(f"Found {test_path.name} without its config record; do not reuse blindly")
            saved = json.loads(config_path.read_text())
            expected = json.loads(json.dumps(dataclasses.asdict(cfg)))
            mismatch = [key for key, value in expected.items() if saved.get(key) != value]
            uncal_path = PRED_DIR / f"{cfg.exp_id}_seed{seed}_test_uncal.csv"
            if mismatch or not uncal_path.is_file():
                raise RuntimeError(
                    f"Existing test output does not match this final configuration (fields: {mismatch}). "
                    "Keep it as the recorded result and choose a new exp_id for any corrected run."
                )
            print("Already evaluated test for this exact config/seed; skipping:", test_path.name)
            continue
        final_runs.append(train.run(cfg))

print("Final/baseline test prediction files:")
for path in sorted(PRED_DIR.glob("*_test.csv")):
    print(path.name, path.stat().st_size)
save_resume_archive()


In [ ]:
# Chạy sau khi predictions đủ cho F01 và T00, mỗi seed 0,1,2 đúng một lần.
subprocess.run([
    sys.executable, str(EVAL_FILE), "score",
    "--pred", str(PRED_DIR / "F01_seed*_test.csv"),
    "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
    "--labels", str(LABELS_DIR / "labels.csv"),
    "--tag", "F01", "--out", str(SUBMISSION_DIR / "eval_out"),
], check=True)

subprocess.run([
    sys.executable, str(EVAL_FILE), "score",
    "--pred", str(PRED_DIR / "T00_seed*_test.csv"),
    "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
    "--labels", str(LABELS_DIR / "labels.csv"),
    "--tag", "T00", "--out", str(SUBMISSION_DIR / "eval_out"),
], check=True)

latency_p95_for_grade = {
    "I00": latency_i00["p95"], "I01": latency_i01["p95"],
    "I02": latency_i02["p95"], "I03_prob": latency_i01["p95"],
    "I03_logit": latency_i01["p95"], "I07": latency_i00["p95"],
}[FINAL_INFERENCE_ID]
subprocess.run([
    sys.executable, str(EVAL_FILE), "grade",
    "--final", str(PRED_DIR / "F01_seed*_test.csv"),
    "--baseline", str(PRED_DIR / "T00_seed*_test.csv"),
    "--uncal", str(PRED_DIR / "F01_seed*_test_uncal.csv"),
    "--final-val", str(PRED_DIR / "F01_seed*_val.csv"),
    "--val-csv", str(LABELS_DIR / "val_subset0.csv"),
    "--latency-p95-ms", str(latency_p95_for_grade),
    "--latency-method", "proper",
    "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
    "--labels", str(LABELS_DIR / "labels.csv"),
    "--out", str(SUBMISSION_DIR / "eval_out"),
], check=True)


## Bước 5 — Tạo results.xlsx

Cell này tổng hợp kết quả thật từ các lần chạy và predictions đã lưu. Kiểm tra xlsx, hoàn thiện báo cáo bằng số liệu sau đó. Không thêm số liệu ước lượng.


In [ ]:
# Tạo các dòng workbook từ artifacts và predictions có thật.
backbone_rows = []
for item in backbone_results:
    backbone_rows.append({
        "exp_id": item["exp_id"], "backbone": item["backbone"],
        "pretrained_tag": item["pretrained_tag"], "params_m": item["params_m"],
        "gmacs": item["gmacs"], "img_size": 224, "epochs": BACKBONE_EPOCHS,
        "seed": item["seed"], "best_epoch": item["best_epoch"],
        "macro_f1_val": item["macro_f1_val"], "top1_val": item["top1_val"],
        "chinee_recall_val": item["chinee_recall_val"],
        "snake_recall_val": item["snake_recall_val"],
        "train_seconds_per_epoch": item["mean_epoch_seconds"],
        "latency_batch1_ms": item.get("latency_batch1_ms"),
        "notes": item.get("gmacs_error"),
    })

change_notes = {
    "T00": ("baseline", "—"),
    "T01": ("A: initialization", "pretrained -> scratch"),
    "T02": ("A: initialization", "fine-tune -> frozen head"),
    "T03": ("B: augmentation", "basic -> CutMix"),
    "T04": ("B: augmentation", "basic -> color jitter"),
    "T05": ("C: loss", "cross-entropy -> focal gamma=2"),
    "T06": ("C: loss/class balance", "CE -> effective-number CE beta=0.999"),
    "T07": ("F: regularization", "EMA off -> decay=0.995"),
    "T08": ("combination", " + ".join(COMBO_SOURCE_IDS)),
}
baseline_macro = next(r["macro_f1_val"] for r in recipe_results if r["exp_id"] == "T00")
training_rows = []
for item in recipe_results:
    axis, change = change_notes[item["exp_id"]]
    val_group = eval_api.load_group(
        str(PRED_DIR / "screen" / f"{item['exp_id']}_seed{item['seed']}_val.csv"),
        str(LABELS_DIR / "val_subset0.csv"), ref_what="val",
    )
    class_f1 = val_group.metrics[0]["f1"]
    rare_class_f1 = (
        f"Chinee Apple={class_f1[dataset.CLASS_NAMES.index('Chinee Apple')]:.4f}; "
        f"Snake Weed={class_f1[dataset.CLASS_NAMES.index('Snake Weed')]:.4f}"
    )
    training_rows.append({
        "exp_id": item["exp_id"], "backbone": item["backbone"],
        "axis": axis, "change_from_T00": change, "seed": item["seed"],
        "macro_f1_val": item["macro_f1_val"], "top1_val": item["top1_val"],
        "chinee_recall_val": item["chinee_recall_val"],
        "snake_recall_val": item["snake_recall_val"],
        "delta_macro_f1_vs_T00": item["macro_f1_val"] - baseline_macro,
        "rare_class_f1": rare_class_f1,
        "notes": f"best_epoch={item['best_epoch']}; ablation_epochs={ABLATION_EPOCHS}",
    })

inference_rows = inference_table.to_dict("records")

test_csv = str(LABELS_DIR / "test_subset0.csv")
val_csv = str(LABELS_DIR / "val_subset0.csv")
final_group = eval_api.load_group(str(PRED_DIR / "F01_seed*_test.csv"), test_csv)
baseline_group = eval_api.load_group(str(PRED_DIR / "T00_seed*_test.csv"), test_csv)
final_val_group = eval_api.load_group(str(PRED_DIR / "F01_seed*_val.csv"), val_csv, ref_what="val")
baseline_val_group = eval_api.load_group(str(PRED_DIR / "T00_seed*_val.csv"), val_csv, ref_what="val")
final_val_by_seed = {pred.seed: metric for pred, metric in zip(final_val_group.preds, final_val_group.metrics)}
baseline_val_by_seed = {pred.seed: metric for pred, metric in zip(baseline_val_group.preds, baseline_val_group.metrics)}
# Summed confusion matrix and a small fixed-seed error gallery for the report.
from sklearn.metrics import ConfusionMatrixDisplay
confusion_sum = sum(metric["confusion"] for metric in final_group.metrics)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(confusion_sum, display_labels=dataset.CLASS_NAMES).plot(
    ax=ax, xticks_rotation=75, colorbar=False, values_format="d"
)
ax.set_title("F01 summed test confusion matrix (3 seeds)")
fig.tight_layout()
fig.savefig(ASSET_DIR / "confusion_F01_test.png", dpi=180, bbox_inches="tight")
plt.show()

first_final = final_group.preds[0]
wrong = np.flatnonzero(first_final.y_pred != first_final.y_true)[:18]
fig, axes = plt.subplots(3, 6, figsize=(18, 10))
for ax, index in zip(axes.flat, wrong):
    filename = first_final.filenames[index]
    with Image.open(IMAGES_DIR / str(filename)) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(
        f"true: {dataset.CLASS_NAMES[int(first_final.y_true[index])]}/"
        f"pred: {dataset.CLASS_NAMES[int(first_final.y_pred[index])]}", fontsize=8
    )
    ax.axis("off")
for ax in list(axes.flat)[len(wrong):]:
    ax.axis("off")
fig.suptitle("F01 test errors, seed " + str(first_final.seed))
fig.tight_layout()
fig.savefig(ASSET_DIR / "error_gallery_F01_seed0.png", dpi=180, bbox_inches="tight")
plt.show()
final_temperatures = {
    seed: json.loads((RUNS_DIR / "F01" / f"seed{seed}" / "summary.json").read_text())["temperature_used"]
    for seed in (0, 1, 2)
}
final_rows = []
per_class_rows = []
for group_name, group in (("F01", final_group), ("T00", baseline_group)):
    macro_mean, macro_std = group.summary["macro_f1"]
    for pred, metric in zip(group.preds, group.metrics):
        final_rows.append({
            "exp_id": group_name, "configuration": (
                f"{BEST_RECIPE_ID} / {FINAL_INFERENCE_ID} + TS(T={final_temperatures[pred.seed]:.4f})"
                if group_name == "F01" else "T00 / I00"
            ),
            "seed": pred.seed,
            "macro_f1_val": (final_val_by_seed if group_name == "F01" else baseline_val_by_seed)[pred.seed]["macro_f1"],
            "macro_f1_test": metric["macro_f1"],
            "top1_test": metric["top1"], "ece_test": metric["ece"],
        })
    final_rows.append({
        "exp_id": group_name, "configuration": "mean ± std",
        "seed": "summary",
        "macro_f1_val": (
            f"{(final_val_group if group_name == 'F01' else baseline_val_group).summary['macro_f1'][0]:.4f} ± "
            f"{(final_val_group if group_name == 'F01' else baseline_val_group).summary['macro_f1'][1]:.4f}"
        ),
        "macro_f1_test": f"{macro_mean:.4f} ± {macro_std:.4f}",
        "top1_test": f"{group.summary['top1'][0]:.4f} ± {group.summary['top1'][1]:.4f}",
        "ece_test": f"{group.summary['ece'][0]:.4f} ± {group.summary['ece'][1]:.4f}",
        "mean_std_summary": f"n={len(group.preds)} seeds; sample std (ddof=1)",
    })
    for class_index, class_name in enumerate(dataset.CLASS_NAMES):
        per_class_rows.append({
            "configuration": group_name, "class": class_name,
            "n_test": int(group.metrics[0]["support"][class_index]),
            "precision": float(group.summary["precision"][0][class_index]),
            "recall": float(group.summary["recall"][0][class_index]),
            "f1": float(group.summary["f1"][0][class_index]),
        })

summary_rows = []
candidate_rows = []
for item in backbone_results + recipe_results:
    candidate_rows.append({
        "exp_id": item["exp_id"], "configuration": item["backbone"],
        "macro_f1_val": item["macro_f1_val"], "top1_val": item["top1_val"],
        "params_m": item["params_m"], "gmacs": item["gmacs"],
        "relative_cost": None, "notes": "training screening",
    })
for item in inference_rows:
    candidate_rows.append({
        "exp_id": item["exp_id"], "configuration": item["method"],
        "macro_f1_val": item["macro_f1_val"], "top1_val": item["top1_val"],
        "p95_ms_batch1": item["p95_ms_batch1"],
        "relative_cost": item["relative_cost_vs_I00"], "notes": item["notes"],
    })
candidate_rows.sort(key=lambda row: row.get("macro_f1_val", 0.0), reverse=True)
for rank, item in enumerate(candidate_rows[:10], start=1):
    summary_rows.append({"rank": rank, **item})

xlsx_path = SUBMISSION_DIR / "results.xlsx"
results.write_results_xlsx(
    xlsx_path,
    backbones=backbone_rows,
    training=training_rows,
    inference=inference_rows,
    final=final_rows,
    per_class=per_class_rows,
    latency=latency_rows,
    summary=summary_rows,
)
print("Workbook saved:", xlsx_path, "| bytes:", xlsx_path.stat().st_size)


## Bước 6 — Xuất artifact để đưa về repo

Tạo một file zip gồm code và đầu ra cần nộp, bỏ checkpoint mô hình/optimizer để tránh commit file lớn. Tải zip từ phần Output của Kaggle, giải nén/merge vào fork rồi cập nhật `report.md` bằng kết quả thật.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
artifact_zip = Path("/kaggle/working/submission_artifacts.zip")
# Chỉ đưa file nhỏ cần cho bài nộp vào Git; checkpoint và runs nằm trong lab_resume.zip.
included_files = [
    SUBMISSION_DIR / name for name in (
        "results.xlsx", "environment.json", "backbone_screen.csv",
        "training_screen.csv", "inference_screen.csv",
    )
]
included_files += list(PRED_DIR.glob("*.csv"))
included_files += list(CURVE_DIR.rglob("*.png"))
included_files += [path for path in ASSET_DIR.rglob("*") if path.is_file() and path.suffix in {".png", ".json"}]
with ZipFile(artifact_zip, "w", compression=ZIP_DEFLATED) as archive:
    for file_path in included_files:
        if not file_path.is_file():
            raise FileNotFoundError(f"Missing submission artifact: {file_path}")
        archive.write(file_path, file_path.relative_to(REPO_DIR))
print("Artifact:", artifact_zip, "| files:", len(included_files), "| MB:", round(artifact_zip.stat().st_size / 1e6, 2))
print("Download this file from the Kaggle notebook Output panel.")